# 📧 Spam Email Classifier — NLP + Scikit-learn + NLTK
### A Beginner-Friendly, Resume-Worthy Machine Learning Project

---

## 🗺️ Project Roadmap
1. **Setup & Imports** — Install and load all libraries
2. **Load Dataset** — Use the classic SMS Spam Collection dataset
3. **Exploratory Data Analysis (EDA)** — Understand the data
4. **Text Preprocessing** — Clean text with NLTK
5. **Feature Engineering** — TF-IDF Vectorization
6. **Model Training** — Train 4 classifiers and compare
7. **Evaluation** — Accuracy, Precision, Recall, F1, Confusion Matrix
8. **Visualization** — Word clouds, charts, heatmaps
9. **Predict on Custom Input** — Test your own emails!
10. **Save & Load Model** — Export the best model with joblib

---
> **Dataset Used:** [UCI SMS Spam Collection](https://archive.ics.uci.edu/ml/datasets/SMS+Spam+Collection) — 5,572 labeled messages (Ham vs Spam)

## 🔧 Step 1: Install Required Libraries
Run this cell once to install everything you need.

In [ ]:
# Run this cell once to install all dependencies
import subprocess
import sys

packages = [
    'nltk',
    'scikit-learn',
    'pandas',
    'numpy',
    'matplotlib',
    'seaborn',
    'wordcloud',
    'joblib'
]

for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

print('✅ All packages installed successfully!')

## 📦 Step 2: Import Libraries & Download NLTK Data

In [ ]:
# ─── Standard Libraries ───────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import re
import string
import warnings
warnings.filterwarnings('ignore')

# ─── Visualization ────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from wordcloud import WordCloud

# ─── NLTK ─────────────────────────────────────────────────────────────────────
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer

# Download required NLTK data
nltk_data = ['punkt', 'stopwords', 'wordnet', 'averaged_perceptron_tagger', 'punkt_tab']
for data in nltk_data:
    nltk.download(data, quiet=True)

# ─── Scikit-learn ─────────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)
from sklearn.pipeline import Pipeline

# ─── Model Saving ─────────────────────────────────────────────────────────────
import joblib

# ─── Style Config ─────────────────────────────────────────────────────────────
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
COLORS = {'ham': '#2ecc71', 'spam': '#e74c3c'}

print('✅ All libraries imported successfully!')
print(f'   pandas  v{pd.__version__}')
print(f'   numpy   v{np.__version__}')
print(f'   sklearn imported')
print(f'   nltk    v{nltk.__version__}')

## 📂 Step 3: Load the Dataset

We use the **SMS Spam Collection Dataset** — a classic NLP benchmark dataset with 5,572 messages labeled as `ham` (legitimate) or `spam`.

We fetch it directly from a public URL so you don't need to download anything manually.

In [ ]:
# ─── Load Dataset ─────────────────────────────────────────────────────────────
# Method 1: Download directly from URL
import urllib.request
import io
import zipfile

url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip'

try:
    print('📥 Downloading dataset from UCI...')
    response = urllib.request.urlopen(url, timeout=10)
    zip_data = io.BytesIO(response.read())
    with zipfile.ZipFile(zip_data) as z:
        with z.open('SMSSpamCollection') as f:
            df = pd.read_csv(f, sep='\t', header=None, names=['label', 'message'])
    print('✅ Dataset loaded from UCI!')
except Exception as e:
    print(f'⚠️  Could not fetch from URL ({e})')
    print('📦 Generating a built-in sample dataset instead...')
    
    # Built-in sample data (100 messages) as fallback
    sample_data = {
        'label': ['ham']*60 + ['spam']*40,
        'message': [
            # HAM messages
            'Hey, are you coming to the party tonight?',
            'I will be home by 8pm, save some dinner for me',
            'Can you pick up some milk on your way home?',
            'The meeting has been rescheduled to 3pm tomorrow',
            'Happy birthday! Hope you have a wonderful day',
            'Did you watch the game last night? It was amazing',
            'I am running 10 minutes late, please wait',
            'Thanks for your help yesterday, really appreciated it',
            'Let me know when you reach the airport',
            'Can we reschedule our lunch to next week?',
            'The project deadline has been moved to Friday',
            'I love the new coffee shop that opened downtown',
            'Do you want to grab lunch tomorrow?',
            'I need to talk to you about something important',
            'The kids are doing great at school this semester',
            'I just finished reading that book you recommended',
            'Can you send me the report when you get a chance?',
            'Let us plan a trip to the mountains next month',
            'My phone battery is dying, will call you back later',
            'The weather forecast says it will rain tomorrow',
            'I got the job offer! Starting next Monday',
            'Please don t forget to water the plants',
            'We need to buy a gift for mom s birthday',
            'The car is ready to be picked up from the garage',
            'Are you free this weekend to help me move?',
            'I will call you when I land at the airport',
            'Can you check if the oven is turned off?',
            'The doctor appointment is at 10am sharp',
            'I sent you the documents, please review them',
            'Dinner was fantastic, thank you for cooking',
            'Please remind me about the 4pm meeting today',
            'I will be working from home for the rest of the week',
            'Did you get my email about the project changes?',
            'The library books are due back by Thursday',
            'I am stuck in traffic, will be late by 20 minutes',
            'What time does the movie start tonight?',
            'I really enjoyed our conversation at the conference',
            'Let me know if you need any help with the assignment',
            'The groceries are in the refrigerator',
            'I finished the report and sent it to the manager',
            'Can we talk on the phone later this evening?',
            'The gym closes at 10pm on weekdays',
            'I will bring the cake for the office party',
            'Your package has been delivered to the front door',
            'I am not feeling well today, might skip the gym',
            'The train to Mumbai leaves at 7am tomorrow',
            'Please don t forget to lock the door tonight',
            'I need your help with the presentation slides',
            'The annual report is ready for your review',
            'Congratulations on your promotion! Well deserved',
            'I just got back from the supermarket',
            'We should catch up over coffee sometime',
            'My internet connection has been very slow today',
            'Can you send me the address of the venue?',
            'I will finish the task before end of day',
            'The kids really enjoyed the school trip today',
            'Let me know if the time works for you',
            'I just subscribed to the new streaming service',
            'The restaurant was fully booked so we went elsewhere',
            'Please let me know your availability for next week',
            # SPAM messages
            'WINNER!! You have won a $1000 Walmart gift card. Call 1-800-WIN NOW!',
            'FREE! Get free iPhone 15 Pro by clicking here: bit.ly/free-iphone',
            'Congratulations! You are selected for a $500 cash prize. Claim now!',
            'URGENT: Your account will be suspended. Click here to verify immediately',
            'You have won a luxury cruise for 2! Reply YES to claim your prize',
            'Make $5000 per week from home. No experience needed. Act now!',
            'Hot singles in your area are waiting to chat. Click here now!',
            'Your loan application has been approved. Get $10000 cash instantly!',
            'SPECIAL OFFER: Lose 30 pounds in 30 days with this miracle pill!',
            'You are the lucky winner of our monthly lottery! Claim your prize now',
            'FREE entry into our prize draw! Text WIN to 80488 now',
            'Congratulations! You ve been selected for exclusive investment opportunity!',
            'Get a free vacation to Bahamas just by filling out this survey!',
            'Act now! Limited time offer. 90% off all designer brands. Shop here',
            'Your PayPal account has been compromised. Click here to secure it now',
            'You have a package waiting! Provide your details to receive it today',
            'FINAL NOTICE: Claim your tax refund of $3200 before it expires!',
            'Earn unlimited income working from home. Join thousands of winners now!',
            'FREE ringtones! Get the latest songs free on your phone. Text now!',
            'Your credit score can be fixed in 24 hours. Call us right away!',
            'Win a brand new car in our sweepstakes. Enter FREE at our website now',
            'CLAIM YOUR PRIZE: You have been selected as our lucky winner today!',
            'Debt relief guaranteed! Get out of debt in 6 months, call us free!',
            'Exclusive deal: Buy 1 get 10 FREE on all products. Click to order!',
            'You are pre-approved for a $50000 loan. Apply online in 2 minutes!',
            'Hot stock tip! Buy shares now before it goes up 500 percent tomorrow!',
            'Your Apple ID has been locked. Verify your details at this link now',
            'CASH PRIZE! You have been randomly selected. Call now to receive $2500',
            'Free gift voucher worth $200 for completing this 30 second survey now',
            'Viagra and Cialis at 80 percent off! No prescription needed. Order now',
            'You won the UK National Lottery! Your ticket matched all 6 numbers',
            'Send this to 10 friends and receive $100 in your PayPal account today',
            'Cheap meds online! Prescription not required. Best prices guaranteed!',
            'LAST CHANCE: Your free Netflix subscription is about to expire. Renew!',
            'Congratulations! Click the link to claim your FREE Amazon gift card now',
            'Your mobile number won! Contact us to claim your prize of 50000 rupees',
            'Earn 2000 rupees per day from home doing simple online work. Join today',
            'URGENT NOTICE: Your bank account shows suspicious activity. Verify now!',
            'Amazing weight loss secret revealed! Lose 20 kg in just one month!',
            'Flash sale! Everything must go. 95 percent off. Limited stock available!'
        ]
    }
    df = pd.DataFrame(sample_data)
    print('✅ Sample dataset created (100 messages).')
    print('   💡 For full results, use the complete UCI dataset (5572 messages).')

# Encode labels: ham=0, spam=1
df['label_num'] = df['label'].map({'ham': 0, 'spam': 1})

print(f'\n📊 Dataset Shape: {df.shape}')
print(f'   Ham  messages: {(df["label"]=="ham").sum()}')
print(f'   Spam messages: {(df["label"]=="spam").sum()}')
df.head(10)

## 🔍 Step 4: Exploratory Data Analysis (EDA)
Understand the data before building any model.

In [ ]:
# ─── Basic Info ───────────────────────────────────────────────────────────────
print('=' * 50)
print('📋 DATASET OVERVIEW')
print('=' * 50)
print(df.info())
print('\n📊 Class Distribution:')
print(df['label'].value_counts())
print(f'\n⚠️  Class Imbalance Ratio: {round(df["label"].value_counts()["ham"] / df["label"].value_counts()["spam"], 2)}:1 (ham:spam)')

In [ ]:
# ─── Add Feature Columns ──────────────────────────────────────────────────────
df['msg_length']    = df['message'].apply(len)
df['word_count']    = df['message'].apply(lambda x: len(x.split()))
df['num_digits']    = df['message'].apply(lambda x: sum(c.isdigit() for c in x))
df['num_uppercase'] = df['message'].apply(lambda x: sum(c.isupper() for c in x))
df['has_url']       = df['message'].apply(lambda x: int(bool(re.search(r'http|www|bit\.ly', x, re.I))))
df['has_currency']  = df['message'].apply(lambda x: int(bool(re.search(r'\$|£|€|₹|rupee', x, re.I))))
df['has_phone']     = df['message'].apply(lambda x: int(bool(re.search(r'\b\d{10}\b|\d{3}[-.]\d{3}[-.]\d{4}', x))))

print('📊 Feature Statistics by Label:')
df.groupby('label')[['msg_length', 'word_count', 'num_digits', 'num_uppercase']].describe().round(2)

In [ ]:
# ─── Visualization: EDA Dashboard ────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('📧 Spam Email Classifier — Exploratory Data Analysis', 
             fontsize=16, fontweight='bold', y=1.02)

ham_df  = df[df['label'] == 'ham']
spam_df = df[df['label'] == 'spam']

# 1. Class Distribution (Pie)
ax1 = axes[0, 0]
counts = df['label'].value_counts()
ax1.pie(counts, labels=counts.index, autopct='%1.1f%%',
        colors=[COLORS['ham'], COLORS['spam']],
        startangle=90, explode=(0.05, 0.05),
        textprops={'fontsize': 12})
ax1.set_title('Class Distribution', fontweight='bold')

# 2. Message Length Distribution
ax2 = axes[0, 1]
ax2.hist(ham_df['msg_length'],  bins=50, alpha=0.7, color=COLORS['ham'],  label='Ham',  edgecolor='white')
ax2.hist(spam_df['msg_length'], bins=50, alpha=0.7, color=COLORS['spam'], label='Spam', edgecolor='white')
ax2.set_xlabel('Message Length (chars)')
ax2.set_ylabel('Count')
ax2.set_title('Message Length Distribution', fontweight='bold')
ax2.legend()

# 3. Word Count Distribution
ax3 = axes[0, 2]
ax3.hist(ham_df['word_count'],  bins=40, alpha=0.7, color=COLORS['ham'],  label='Ham',  edgecolor='white')
ax3.hist(spam_df['word_count'], bins=40, alpha=0.7, color=COLORS['spam'], label='Spam', edgecolor='white')
ax3.set_xlabel('Word Count')
ax3.set_ylabel('Count')
ax3.set_title('Word Count Distribution', fontweight='bold')
ax3.legend()

# 4. Uppercase Characters
ax4 = axes[1, 0]
ax4.boxplot([ham_df['num_uppercase'], spam_df['num_uppercase']],
            labels=['Ham', 'Spam'],
            patch_artist=True,
            boxprops=dict(facecolor='lightblue'),
            medianprops=dict(color='red', linewidth=2))
ax4.set_ylabel('Uppercase Count')
ax4.set_title('Uppercase Characters (Ham vs Spam)', fontweight='bold')

# 5. Has URL vs Spam
ax5 = axes[1, 1]
url_counts = df.groupby(['label', 'has_url']).size().unstack(fill_value=0)
url_counts.plot(kind='bar', ax=ax5, color=['#95a5a6', '#e74c3c'], 
                edgecolor='white', rot=0)
ax5.set_xlabel('Label')
ax5.set_ylabel('Count')
ax5.set_title('Messages with URLs', fontweight='bold')
ax5.legend(['No URL', 'Has URL'], loc='upper right')

# 6. Average Message Length Bar Chart
ax6 = axes[1, 2]
avg_lengths = df.groupby('label')[['msg_length', 'word_count', 'num_uppercase']].mean()
avg_lengths.T.plot(kind='bar', ax=ax6, color=[COLORS['ham'], COLORS['spam']], 
                   edgecolor='white', rot=15)
ax6.set_title('Average Features by Label', fontweight='bold')
ax6.set_ylabel('Average Value')
ax6.legend(['Ham', 'Spam'])

plt.tight_layout()
plt.savefig('eda_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ EDA chart saved as eda_analysis.png')

## 🧹 Step 5: Text Preprocessing with NLTK

This is the most important NLP step. We will:
- Lowercase the text
- Remove punctuation, numbers, special characters
- Tokenize (split into words)
- Remove stopwords ("the", "is", "a", etc.)
- Apply stemming/lemmatization

In [ ]:
# ─── Initialize NLP Tools ────────────────────────────────────────────────────
stemmer     = PorterStemmer()
lemmatizer  = WordNetLemmatizer()
stop_words  = set(stopwords.words('english'))

# Add custom spam-related stop words that are too generic to help
custom_stops = {'u', 'ur', 'r', 'll', 'n', '2', 'gt', 'lt'}
stop_words.update(custom_stops)

def preprocess_text(text, use_lemmatize=True):
    """
    Full NLP preprocessing pipeline:
    1. Lowercase
    2. Remove URLs
    3. Remove special characters & numbers
    4. Remove punctuation
    5. Tokenize
    6. Remove stopwords
    7. Lemmatize or Stem
    """
    # Step 1: Lowercase
    text = text.lower()
    
    # Step 2: Remove URLs
    text = re.sub(r'http\S+|www\S+|bit\.ly\S+', ' url ', text)
    
    # Step 3: Remove phone numbers
    text = re.sub(r'\b\d{10,}\b|\d{3}[-.]\d{3}[-.]\d{4}', ' phone ', text)
    
    # Step 4: Replace currency symbols
    text = re.sub(r'\$|£|€|₹', ' money ', text)
    
    # Step 5: Remove special characters and numbers (keep letters + spaces)
    text = re.sub(r'[^a-z\s]', ' ', text)
    
    # Step 6: Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Step 7: Tokenize
    tokens = word_tokenize(text)
    
    # Step 8: Remove stopwords & short tokens (< 2 chars)
    tokens = [t for t in tokens if t not in stop_words and len(t) > 2]
    
    # Step 9: Lemmatize or Stem
    if use_lemmatize:
        tokens = [lemmatizer.lemmatize(t) for t in tokens]
    else:
        tokens = [stemmer.stem(t) for t in tokens]
    
    return ' '.join(tokens)

# Apply preprocessing
print('⏳ Preprocessing messages...')
df['clean_message'] = df['message'].apply(preprocess_text)
print('✅ Preprocessing complete!')

# Show before/after comparison
print('\n' + '='*60)
print('📝 BEFORE vs AFTER Preprocessing:')
print('='*60)
for i, row in df.sample(4, random_state=42).iterrows():
    print(f'\n[{row["label"].upper()}]')
    print(f'  BEFORE: {row["message"][:100]}...' if len(row['message']) > 100 else f'  BEFORE: {row["message"]}')
    print(f'  AFTER:  {row["clean_message"]}')

## ☁️ Step 6: Word Cloud Visualization

In [ ]:
# ─── Word Clouds ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('Most Frequent Words in Ham vs Spam Messages', 
             fontsize=15, fontweight='bold')

for ax, label, color, title in zip(
    axes,
    ['ham', 'spam'],
    ['#1a6b3c', '#8b0000'],
    ['💚 HAM (Legitimate) Messages', '🔴 SPAM Messages']
):
    text_corpus = ' '.join(df[df['label'] == label]['clean_message'])
    
    if len(text_corpus.strip()) == 0:
        ax.text(0.5, 0.5, 'Not enough data', ha='center', va='center')
    else:
        wc = WordCloud(
            width=800, height=400,
            background_color='white',
            colormap='Greens' if label == 'ham' else 'Reds',
            max_words=100,
            collocations=False,
            prefer_horizontal=0.7
        ).generate(text_corpus)
        
        ax.imshow(wc, interpolation='bilinear')
    
    ax.set_title(title, fontsize=13, fontweight='bold', pad=12)
    ax.axis('off')

plt.tight_layout()
plt.savefig('wordclouds.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Word clouds saved as wordclouds.png')

In [ ]:
# ─── Top 20 Words for Each Class ──────────────────────────────────────────────
from collections import Counter

def get_top_words(df, label, n=20):
    words = ' '.join(df[df['label'] == label]['clean_message']).split()
    return Counter(words).most_common(n)

ham_words  = get_top_words(df, 'ham')
spam_words = get_top_words(df, 'spam')

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('Top 20 Most Frequent Words', fontsize=14, fontweight='bold')

for ax, words, color, title in zip(
    axes,
    [ham_words, spam_words],
    ['#2ecc71', '#e74c3c'],
    ['Top Ham Words', 'Top Spam Words']
):
    words_list, counts = zip(*words) if words else ([], [])
    bars = ax.barh(range(len(words_list)), counts, color=color, edgecolor='white', alpha=0.85)
    ax.set_yticks(range(len(words_list)))
    ax.set_yticklabels(words_list, fontsize=11)
    ax.invert_yaxis()
    ax.set_xlabel('Frequency')
    ax.set_title(title, fontweight='bold')
    
    # Add count labels
    for bar, count in zip(bars, counts):
        ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2, 
                str(count), va='center', fontsize=9)

plt.tight_layout()
plt.savefig('top_words.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Top words chart saved as top_words.png')

## ⚙️ Step 7: Feature Engineering — TF-IDF Vectorization

**TF-IDF** (Term Frequency - Inverse Document Frequency) converts text into numbers.
- **TF**: How often a word appears in a document
- **IDF**: How rare/common a word is across all documents
- Words that are common everywhere (like "the") get low scores, but rare important words get high scores.

In [ ]:
# ─── Train/Test Split ──────────────────────────────────────────────────────
X = df['clean_message']
y = df['label_num']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y          # ensures same class ratio in train & test
)

print(f'📊 Train/Test Split:')
print(f'   Training:   {len(X_train)} samples')
print(f'   Testing:    {len(X_test)} samples')
print(f'   Train Spam: {y_train.sum()} | Ham: {(y_train==0).sum()}')
print(f'   Test  Spam: {y_test.sum()}  | Ham: {(y_test==0).sum()}')

# ─── TF-IDF Vectorizer ────────────────────────────────────────────────────────
tfidf = TfidfVectorizer(
    max_features=5000,     # Use top 5000 words by TF-IDF score
    ngram_range=(1, 2),    # Include unigrams and bigrams
    min_df=2,              # Word must appear in at least 2 documents
    sublinear_tf=True      # Apply log normalization to TF
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

print(f'\n✅ TF-IDF Vectorization complete!')
print(f'   Vocabulary size:      {len(tfidf.vocabulary_)}')
print(f'   Training matrix:      {X_train_tfidf.shape}')
print(f'   Test matrix:          {X_test_tfidf.shape}')
print(f'   Matrix sparsity:      {100 * (1 - X_train_tfidf.nnz / (X_train_tfidf.shape[0] * X_train_tfidf.shape[1])):.2f}%')

## 🤖 Step 8: Train Multiple Classifiers

We train 4 different classifiers to compare their performance:

| Model | Why It's Great for NLP |
|-------|------------------------|
| **Naive Bayes** | Classic spam filter; fast, works well with word frequencies |
| **Logistic Regression** | Strong baseline; interpretable coefficients |
| **Linear SVM** | Excellent for high-dimensional text data |
| **Random Forest** | Ensemble method; handles non-linear patterns |

In [ ]:
# ─── Define Models ────────────────────────────────────────────────────────────
models = {
    'Naive Bayes':         MultinomialNB(alpha=0.1),
    'Logistic Regression': LogisticRegression(C=1.0, max_iter=1000, random_state=42),
    'Linear SVM':          LinearSVC(C=1.0, max_iter=2000, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
}

# ─── Train & Evaluate All Models ──────────────────────────────────────────────
results = {}

print('🏋️  Training and evaluating all models...\n')
print(f'{"Model":<22} {"Accuracy":>10} {"Precision":>11} {"Recall":>9} {"F1 Score":>10}')
print('─' * 66)

for name, model in models.items():
    # Train
    model.fit(X_train_tfidf, y_train)
    
    # Predict
    y_pred = model.predict(X_test_tfidf)
    
    # Calculate metrics
    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec  = recall_score(y_test, y_pred)
    f1   = f1_score(y_test, y_pred)
    
    results[name] = {
        'model':     model,
        'y_pred':    y_pred,
        'accuracy':  acc,
        'precision': prec,
        'recall':    rec,
        'f1':        f1
    }
    
    print(f'{name:<22} {acc:>10.4f} {prec:>11.4f} {rec:>9.4f} {f1:>10.4f}')

print('─' * 66)
best_model_name = max(results, key=lambda x: results[x]['f1'])
print(f'\n🏆 Best Model (by F1 Score): {best_model_name} ({results[best_model_name]["f1"]:.4f})')

## 📊 Step 9: Model Evaluation & Visualization

In [ ]:
# ─── Model Comparison Chart ───────────────────────────────────────────────────
metrics  = ['accuracy', 'precision', 'recall', 'f1']
x        = np.arange(len(metrics))
n_models = len(models)
width    = 0.18
colors_bar = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12']

fig, ax = plt.subplots(figsize=(14, 6))

for i, (name, res) in enumerate(results.items()):
    vals = [res[m] for m in metrics]
    offset = (i - n_models/2 + 0.5) * width
    bars = ax.bar(x + offset, vals, width, label=name, color=colors_bar[i], 
                  alpha=0.85, edgecolor='white')
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=7.5)

ax.set_xlabel('Metric', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([m.capitalize() for m in metrics], fontsize=11)
ax.set_ylim(0.5, 1.08)
ax.legend(loc='lower right', fontsize=10)
ax.axhline(y=1.0, color='gray', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Model comparison chart saved.')

In [ ]:
# ─── Confusion Matrices for All Models ───────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 11))
fig.suptitle('Confusion Matrices — All Models', fontsize=15, fontweight='bold')

for ax, (name, res) in zip(axes.flatten(), results.items()):
    cm = confusion_matrix(y_test, res['y_pred'])
    
    sns.heatmap(
        cm, annot=True, fmt='d', ax=ax,
        cmap='Blues',
        xticklabels=['Ham (0)', 'Spam (1)'],
        yticklabels=['Ham (0)', 'Spam (1)'],
        linewidths=0.5, linecolor='gray',
        annot_kws={'size': 14, 'weight': 'bold'}
    )
    
    tn, fp, fn, tp = cm.ravel()
    ax.set_title(f'{name}\nAcc: {res["accuracy"]:.3f} | F1: {res["f1"]:.3f}', 
                 fontweight='bold', fontsize=11)
    ax.set_ylabel('Actual Label')
    ax.set_xlabel('Predicted Label')
    
    # Annotations
    ax.text(1.1, -0.1, f'TP={tp} | FP={fp}\nTN={tn} | FN={fn}', 
            transform=ax.transAxes, fontsize=8, color='gray', va='top')

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Confusion matrices saved.')

In [ ]:
# ─── Detailed Classification Report for Best Model ───────────────────────────
best = results[best_model_name]

print(f'\n{'='*55}')
print(f'📋 DETAILED REPORT — {best_model_name}')
print('='*55)
print(classification_report(
    y_test, best['y_pred'],
    target_names=['Ham (Legitimate)', 'Spam'],
    digits=4
))

# Error Analysis
test_df = pd.DataFrame({
    'message':    X_test.values,
    'actual':     y_test.values,
    'predicted':  best['y_pred']
})

false_positives = test_df[(test_df['actual']==0) & (test_df['predicted']==1)]
false_negatives = test_df[(test_df['actual']==1) & (test_df['predicted']==0)]

print(f'\n⚠️  Error Analysis:')
print(f'   False Positives (Ham predicted as Spam): {len(false_positives)}')
print(f'   False Negatives (Spam predicted as Ham): {len(false_negatives)}')

if len(false_positives) > 0:
    print(f'\n❌ Example False Positives (Ham → wrongly marked Spam):')
    for _, row in false_positives.head(3).iterrows():
        print(f'   "{row["message"][:80]}..."' if len(row['message']) > 80 else f'   "{row["message"]}"')

if len(false_negatives) > 0:
    print(f'\n❌ Example False Negatives (Spam → slipped through as Ham):')
    for _, row in false_negatives.head(3).iterrows():
        print(f'   "{row["message"][:80]}..."' if len(row['message']) > 80 else f'   "{row["message"]}"')

## 🔑 Step 10: Feature Importance — What Words Indicate Spam?

In [ ]:
# ─── Top Spam/Ham Indicator Words (Logistic Regression) ──────────────────────
lr_model = results['Logistic Regression']['model']
feature_names = np.array(tfidf.get_feature_names_out())

# Get coefficients
coef = lr_model.coef_[0]

# Top spam indicators (high positive coefficients)
top_spam_idx = coef.argsort()[-20:][::-1]
top_ham_idx  = coef.argsort()[:20]

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('Top Feature Words (Logistic Regression Coefficients)', 
             fontsize=14, fontweight='bold')

# Spam-leaning words
ax1 = axes[0]
spam_words_feat = feature_names[top_spam_idx]
spam_coefs      = coef[top_spam_idx]
bars = ax1.barh(range(len(spam_words_feat)), spam_coefs, color='#e74c3c', alpha=0.8, edgecolor='white')
ax1.set_yticks(range(len(spam_words_feat)))
ax1.set_yticklabels(spam_words_feat, fontsize=10)
ax1.invert_yaxis()
ax1.set_title('🔴 Top SPAM Indicator Words', fontweight='bold')
ax1.set_xlabel('Coefficient Value')
ax1.axvline(x=0, color='black', linewidth=0.5)

# Ham-leaning words
ax2 = axes[1]
ham_words_feat = feature_names[top_ham_idx]
ham_coefs      = coef[top_ham_idx]
bars = ax2.barh(range(len(ham_words_feat)), np.abs(ham_coefs), color='#2ecc71', alpha=0.8, edgecolor='white')
ax2.set_yticks(range(len(ham_words_feat)))
ax2.set_yticklabels(ham_words_feat, fontsize=10)
ax2.invert_yaxis()
ax2.set_title('💚 Top HAM Indicator Words', fontweight='bold')
ax2.set_xlabel('|Coefficient Value|')

plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Feature importance chart saved.')

## 🧪 Step 11: Cross-Validation (More Robust Evaluation)

In [ ]:
# ─── 5-Fold Cross Validation ─────────────────────────────────────────────────
from sklearn.pipeline import Pipeline

print('🔄 Running 5-Fold Cross-Validation on All Models...\n')
print(f'{"Model":<22} {"Mean CV Acc":>12} {"Std Dev":>10}')
print('─' * 46)

cv_results = {}

# Build full pipeline (vectorizer + model)
cv_models = {
    'Naive Bayes':         MultinomialNB(alpha=0.1),
    'Logistic Regression': LogisticRegression(C=1.0, max_iter=1000, random_state=42),
    'Linear SVM':          LinearSVC(C=1.0, max_iter=2000, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=50, random_state=42)
}

tfidf_cv = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=2, sublinear_tf=True)

for name, model in cv_models.items():
    pipe = Pipeline([
        ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1,2), min_df=2, sublinear_tf=True)),
        ('clf',   model)
    ])
    
    scores = cross_val_score(pipe, X, y, cv=5, scoring='f1', n_jobs=-1)
    cv_results[name] = scores
    print(f'{name:<22} {scores.mean():>12.4f} {scores.std():>10.4f}')

print('─' * 46)
print('\n✅ Cross-validation complete!')

# Box plot of CV scores
fig, ax = plt.subplots(figsize=(10, 5))
bp = ax.boxplot(
    [cv_results[name] for name in cv_results],
    labels=list(cv_results.keys()),
    patch_artist=True,
    medianprops=dict(color='black', linewidth=2)
)
for patch, color in zip(bp['boxes'], colors_bar):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_title('5-Fold Cross-Validation F1 Scores', fontsize=13, fontweight='bold')
ax.set_ylabel('F1 Score')
ax.set_ylim(0.5, 1.05)
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig('cross_validation.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ CV chart saved.')

## 🎯 Step 12: Build a Prediction Pipeline (The Best Model)

Now let's build a clean, end-to-end prediction function using the best model.

In [ ]:
# ─── Build the Final Pipeline ────────────────────────────────────────────────
best_clf = results[best_model_name]['model']

def predict_spam(message, return_prob=True):
    """
    Classify a message as spam or ham.
    
    Parameters:
        message (str): Raw email/SMS text
        return_prob (bool): Whether to return confidence score
    
    Returns:
        dict: prediction, label, confidence
    """
    # Preprocess
    cleaned = preprocess_text(message)
    
    # Vectorize
    vectorized = tfidf.transform([cleaned])
    
    # Predict
    prediction = best_clf.predict(vectorized)[0]
    label      = 'SPAM 🚨' if prediction == 1 else 'HAM ✅'
    
    # Confidence (if supported)
    confidence = None
    if return_prob and hasattr(best_clf, 'predict_proba'):
        proba      = best_clf.predict_proba(vectorized)[0]
        confidence = proba[prediction]
    elif return_prob and hasattr(best_clf, 'decision_function'):
        score      = best_clf.decision_function(vectorized)[0]
        confidence = 1 / (1 + np.exp(-abs(score)))  # sigmoid
    
    return {
        'message':    message[:60] + '...' if len(message) > 60 else message,
        'label':      label,
        'prediction': prediction,
        'confidence': f'{confidence*100:.1f}%' if confidence else 'N/A',
        'cleaned':    cleaned
    }

# ─── Test Messages ───────────────────────────────────────────────────────────
test_messages = [
    'Congratulations! You won a $1000 Amazon gift card. Click here NOW to claim!',
    'Hey, are you free for lunch tomorrow? I know a great new place.',
    'URGENT: Your bank account has been suspended. Verify immediately at this link!',
    'The team meeting has been moved to 3pm. Please update your calendar.',
    'FREE iPhone 15! You have been selected as our lucky winner. Reply YES now!',
    'Don t forget to bring the documents for tomorrow s interview. Good luck!',
    'Make $5000 a week from home! No experience needed! Act now before offer expires!',
    'I will be late to the office today. Please start the standup without me.'
]

print('='*65)
print(f'🔍 SPAM DETECTOR — Using {best_model_name}')
print('='*65)

for msg in test_messages:
    result = predict_spam(msg)
    print(f'\n📩 "{result["message"]}"')
    print(f'   → {result["label"]}  (Confidence: {result["confidence"]})')

## ✍️ Step 13: Try Your Own Email!
Type any message and see if it's spam or not.

In [ ]:
# ─── Interactive Spam Checker ────────────────────────────────────────────────
# Change the message below and run this cell!

your_message = """You have been pre-approved for a personal loan of Rs 5,00,000! 
No documents required. Apply now and get money in your account within 24 hours. 
Click here: bit.ly/instant-loan-2024"""

# ─────────────────────────────────────────────────────────────────────────────
result = predict_spam(your_message)

print('\n' + '🔮'*20)
print('\n📧 YOUR MESSAGE:')
print(f'   {your_message[:120]}...' if len(your_message) > 120 else f'   {your_message}')
print(f'\n🎯 PREDICTION:  {result["label"]}')
print(f'📊 CONFIDENCE:  {result["confidence"]}')
print(f'🧹 CLEANED:     {result["cleaned"][:100]}...' if len(result['cleaned']) > 100 else f'🧹 CLEANED:     {result["cleaned"]}')
print('\n' + '🔮'*20)

## 💾 Step 14: Save & Load the Model

In [ ]:
# ─── Save the Model and Vectorizer ───────────────────────────────────────────
joblib.dump(best_clf, 'spam_classifier_model.pkl')
joblib.dump(tfidf,    'tfidf_vectorizer.pkl')

print('✅ Model saved to:      spam_classifier_model.pkl')
print('✅ Vectorizer saved to: tfidf_vectorizer.pkl')

# ─── Load and Test ────────────────────────────────────────────────────────────
loaded_model    = joblib.load('spam_classifier_model.pkl')
loaded_tfidf    = joblib.load('tfidf_vectorizer.pkl')

# Quick test with loaded model
test_msg   = 'WIN A FREE iPHONE 15! Click here to claim your prize now!!!'
test_clean = preprocess_text(test_msg)
test_vec   = loaded_tfidf.transform([test_clean])
test_pred  = loaded_model.predict(test_vec)[0]

print(f'\n🔁 Test with loaded model:')
print(f'   Message: "{test_msg}"')
print(f'   Result:  {"SPAM 🚨" if test_pred == 1 else "HAM ✅"}')

## 📈 Step 15: Final Summary & Resume Bullet Points

In [ ]:
# ─── Final Project Summary ────────────────────────────────────────────────────
best_acc = results[best_model_name]['accuracy'] * 100
best_f1  = results[best_model_name]['f1'] * 100
best_prec = results[best_model_name]['precision'] * 100
best_rec  = results[best_model_name]['recall'] * 100

print('\n' + '🎉'*25)
print('\n📋 PROJECT COMPLETE — FINAL SUMMARY')
print('='*60)
print(f'\n📦 Dataset:        SMS Spam Collection ({len(df)} messages)')
print(f'🧹 Preprocessing:  NLTK (tokenize, stopwords, lemmatize)')
print(f'🔢 Features:       TF-IDF (5000 features, 1-2 grams)')
print(f'🤖 Models Trained: {len(models)} classifiers')
print(f'🏆 Best Model:     {best_model_name}')
print(f'   ├─ Accuracy:   {best_acc:.2f}%')
print(f'   ├─ Precision:  {best_prec:.2f}%')
print(f'   ├─ Recall:     {best_rec:.2f}%')
print(f'   └─ F1 Score:   {best_f1:.2f}%')

print('\n' + '='*60)
print('\n💼 RESUME BULLET POINTS (Copy-Paste Ready!):')
print('─'*60)
print(f'''• Built a Spam Email Classifier using NLP and Scikit-learn, achieving
  {best_acc:.1f}% accuracy and {best_f1:.1f}% F1-Score on the SMS Spam Collection dataset.

• Developed a complete NLP preprocessing pipeline using NLTK including
  tokenization, stopword removal, and lemmatization.

• Implemented and compared 4 ML classifiers (Naive Bayes, Logistic Regression,
  Linear SVM, Random Forest) using TF-IDF feature extraction with n-grams.

• Conducted thorough model evaluation using 5-fold cross-validation, confusion
  matrices, and feature importance analysis to identify key spam indicators.

• Deployed the trained model using joblib for persistence and created an
  end-to-end prediction pipeline for real-world usage.
''')

print('\n📁 Files Generated:')
files = [
    'spam_classifier_model.pkl  → Trained ML model',
    'tfidf_vectorizer.pkl       → TF-IDF vectorizer',
    'eda_analysis.png           → EDA dashboard',
    'wordclouds.png             → Ham vs Spam word clouds',
    'top_words.png              → Top 20 words per class',
    'model_comparison.png       → Model performance comparison',
    'confusion_matrices.png     → Confusion matrices',
    'feature_importance.png     → Feature importance chart',
    'cross_validation.png       → CV scores boxplot'
]
for f in files:
    print(f'   ✅ {f}')

print('\n' + '🎉'*25)